# Modifying for W2C BOICL: Imports - This also ensures that we are working in the local repositories/python files, so changes will be reflected in this jnotebook

In [11]:
# # import matplotlib.pyplot as plt
# from langchain.prompts.prompt import PromptTemplate
# import copy, cloudpickle
# import sys
# import os
# current_dir = os.getcwd()
# parent_dir = os.path.join(current_dir,'..')
# sys.path.insert(0, parent_dir)
# import boicl
# from dotenv import load_dotenv
# load_dotenv()

from langchain.prompts.prompt import PromptTemplate
import copy, cloudpickle
import sys
import os

# Load env FIRST before boicl tries to instantiate OpenAI client
from dotenv import load_dotenv
load_dotenv()

# Force local boicl over any PyPI install
sys.path.insert(0, "./repos/boicl_crystal_phase_isolation")

import boicl
print("boicl loaded from:", boicl.__file__)

/Users/shane/opt/anaconda3/envs/bo_icl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


boicl loaded from: /Users/shane/repos/boicl_crystal_phase_isolation/boicl/__init__.py


# Plotting formatt

In [2]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as font_manager
import urllib.request

urllib.request.urlretrieve(
    "https://github.com/google/fonts/raw/main/ofl/ibmplexmono/IBMPlexMono-Regular.ttf",
    "IBMPlexMono-Regular.ttf",
)
fe = font_manager.FontEntry(fname="IBMPlexMono-Regular.ttf", name="plexmono")
font_manager.fontManager.ttflist.append(fe)
plt.rcParams.update(
    {
        "axes.facecolor": "#f5f4e9",
        "grid.color": "#AAAAAA",
        "axes.edgecolor": "#333333",
        "figure.facecolor": "#FFFFFF",
        "axes.grid": False,
        "axes.prop_cycle": plt.cycler("color", plt.cm.Dark2.colors),
        "font.family": fe.name,
        "figure.figsize": (3.5, 3.5 / 1.2),
        "ytick.left": True,
        "xtick.bottom": True,
    }
)


# Trajector 1: EI

## Dataset prep : find completion indices for tell, remove completed prompts from unlabled pool

In [3]:
import pandas as pd
import numpy as np
import time

start_time = time.time()
RANDOM_SEED = 616
np.random.seed(RANDOM_SEED)

# Constants for column names
PROMPT_COL     = "Procedure"
COMPLETION_COL = "MoC(Fm3m) weight fraction"

# ── New results to write in each BO iteration ─────────────────────────────────
# Format: (procedure_string, Mo_wf, Mo_sigma, Mo2C_wf, Mo2C_sigma,
#           MoC_wf, MoC_sigma, MoO2_wf, MoO2_sigma,
#           Rwp, GOF, chi2, MoC_ratio)
completed_prompt_labels = [
    ("1 g of ammonium heptamolybdate tetrahydrate dissolved in 5.5 mL of DI water. Separately, 1 g of sucrose dissolved in 2 mL of DI water at 50 °C. The two solutions were then combined. The combined solution was dried at 120 °C in air for 24 hours, then crushed and sieved to 212 microns. 0.6 g of the sieved precursor was carburized at a ramp rate of 10 °C/min to 620 °C for 10 hr under 60 sccm of N2 gas. After carburization the sample was cooled under 30 sccm N2 and passivated under 30 sccm 1% O2/N2 for 2 hours.",
     0.0, 0.0,           # Mo wf, sigma
     0.0, 0.0,           # Mo2C wf, sigma
     86.7, 1.01,         # MoC wf, sigma
     13.3, 3.71,         # MoO2 wf, sigma
     8.41, 12.853, 165.21,  # Rwp, GOF, chi2
     "MoC.66"),("1 g of ammonium heptamolybdate tetrahydrate dissolved in 5.5 mL of DI water. Separately, 1 g of sucrose dissolved in 2 mL of DI water at 50 °C. The two solutions were then combined. The combined solution was dried at 120 °C in air for 24 hours, then crushed and sieved to 212 microns. 0.6 g of the sieved precursor was carburized at a ramp rate of 15 °C/min to 620 °C for 0.5 hr under 30 sccm of H2 gas. After carburization the sample was cooled under 30 sccm N2 and passivated under 30 sccm 1% O2/N2 for 2 hours.",0.0, 0.0,           # Mo wf, sigma
     0.0, 0.0,           # Mo2C wf, sigma
     91.7, 1.83,         # MoC wf, sigma
     8.3, 1.83,         # MoO2 wf, sigma
     7, 11.032, 121.71,  # Rwp, GOF, chi2
     "MoC.66"),("1 g of ammonium heptamolybdate tetrahydrate dissolved in 5.5 mL of DI water. Separately, 1 g of sucrose dissolved in 2 mL of DI water at 50 °C. The two solutions were then combined. The combined solution was dried at 120 °C in air for 24 hours, then crushed and sieved to 212 microns. 0.6 g of the sieved precursor was carburized at a ramp rate of 10 °C/min to 620 °C for 0.5 hr under 60 sccm of N2 gas. After carburization the sample was cooled under 30 sccm N2 and passivated under 30 sccm 1% O2/N2 for 2 hours.",0.0, 0.0,           # Mo wf, sigma
     0.0, 0.0,           # Mo2C wf, sigma
     91.8, 5.61,         # MoC wf, sigma
     8.2, 5.61,         # MoO2 wf, sigma
     4.0, 1.4, 1.87,  # Rwp, GOF, chi2
     "MoC.66")
]

# Load dataset
data_path = "./Metal_Sucrose_DesignSpace_fr.xlsx"
df = pd.read_excel(data_path, sheet_name="Design Space")

# ── Add new columns if they don't exist ───────────────────────────────────────
for col in ["MoO2(P21/c) weight fraction", "sigma.3", "MoC Ratio"]:
    if col not in df.columns:
        df[col] = None
        print(f"➕ Added column: {col}")

# ── Write new results ─────────────────────────────────────────────────────────
for row in completed_prompt_labels:
    (procedure, mo_wf, mo_sig, mo2c_wf, mo2c_sig,
     moc_wf, moc_sig, moo2_wf, moo2_sig,
     rwp, gof, chi2, moc_ratio) = row
    match = df[df[PROMPT_COL] == procedure]
    if not match.empty:
        idx = match.index[0]
        df.at[idx, 'Mo (Im3m) weight fraction ']     = mo_wf
        df.at[idx, 'sigma']                           = mo_sig
        df.at[idx, 'Mo2C(pbcn) weight fraction']      = mo2c_wf
        df.at[idx, 'sigma.1']                         = mo2c_sig
        df.at[idx, 'MoC(Fm3m) weight fraction']       = moc_wf
        df.at[idx, 'sigma.2']                         = moc_sig
        df.at[idx, 'MoO2(P21/c) weight fraction']     = moo2_wf
        df.at[idx, 'sigma.3']                         = moo2_sig
        df.at[idx, 'Rwp']                             = rwp
        df.at[idx, 'GOF']                             = gof
        df.at[idx, 'X^2']                             = chi2
        df.at[idx, 'MoC Ratio']                       = moc_ratio
        print(f"✅ Updated index {idx} — MoC wf={moc_wf}, MoO2 wf={moo2_wf}, Rwp={rwp}")
    else:
        print(f"❌ Prompt not found: {procedure[:60]}...")

# Save back to xlsx
with pd.ExcelWriter(data_path, engine="openpyxl", mode="a",
                    if_sheet_exists="replace") as writer:
    df.to_excel(writer, sheet_name="Design Space", index=False)

# ── Shuffle and report non-zero entries ───────────────────────────────────────
df_shuffled = df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

nonzero_mask    = df_shuffled[COMPLETION_COL].notna() & \
                  (df_shuffled[COMPLETION_COL].astype(str).str.strip() != "")
nonzero_indices = df_shuffled.index[nonzero_mask].to_numpy()

print("Non-zero indices:", nonzero_indices[:])
print(f"Total non-zero entries: {len(nonzero_indices)}")
print(f"\n⏱️ Elapsed time: {time.time() - start_time:.2f} seconds")

✅ Updated index 1642 — MoC wf=86.7, MoO2 wf=13.3, Rwp=8.41
✅ Updated index 1668 — MoC wf=91.7, MoO2 wf=8.3, Rwp=7
✅ Updated index 1633 — MoC wf=91.8, MoO2 wf=8.2, Rwp=4.0
Non-zero indices: [1226 2345 3331 4251 5846 6865]
Total non-zero entries: 6

⏱️ Elapsed time: 3.53 seconds


In [4]:
for i in nonzero_indices:
    print(df_shuffled[COMPLETION_COL][i])

91.7
86.7
23.8
74.4
91.8
76.3


In [5]:
# Ensure MoC weight fraction is numeric
df_shuffled[COMPLETION_COL] = pd.to_numeric(df_shuffled[COMPLETION_COL], errors="coerce")

# Row with the highest MoC cubic weight fraction (best observed so far)
index_max = df_shuffled[COMPLETION_COL].idxmax()
print(f"Row index with max MoC wf: {index_max}")
print(f"Max MoC wf: {df_shuffled.at[index_max, COMPLETION_COL]:.3f}")
print(f"Procedure: {df_shuffled.at[index_max, PROMPT_COL]}")
print(f"DataFrame length: {len(df_shuffled)}")

Row index with max MoC wf: 5846
Max MoC wf: 91.800
Procedure: 1 g of ammonium heptamolybdate tetrahydrate dissolved in 5.5 mL of DI water. Separately, 1 g of sucrose dissolved in 2 mL of DI water at 50 °C. The two solutions were then combined. The combined solution was dried at 120 °C in air for 24 hours, then crushed and sieved to 212 microns. 0.6 g of the sieved precursor was carburized at a ramp rate of 10 °C/min to 620 °C for 0.5 hr under 60 sccm of N2 gas. After carburization the sample was cooled under 30 sccm N2 and passivated under 30 sccm 1% O2/N2 for 2 hours.
DataFrame length: 7779


In [6]:
system_message_path     = "./BOICL_docs/pred_system_message.txt"
inv_system_message_path = "./BOICL_docs/inv_system_message.txt"

assert os.path.exists(system_message_path), f"Missing: {system_message_path}"
assert os.path.exists(inv_system_message_path), f"Missing: {inv_system_message_path}"

with open(system_message_path, "r") as f:
    system_message = f.read()

with open(inv_system_message_path, "r") as f:
    inv_system_message = f.read()

print("✅ System messages loaded")
print(f"  pred: {len(system_message)} chars")
print(f"  inv:  {len(inv_system_message)} chars")

# ── Original simple asktell (string x, no phase context) ─────────────────────
# asktell = boicl.AskTellFewShotTopk(
#     prefix="",
#     prompt_template=PromptTemplate(
#         input_variables=["x", "y", "y_name"],
#         template="Q: What is the {y_name} of {x}?@@@\nA: {y}###",
#     ),
#     suffix="What is the {y_name} of {x}?@@@\nA:",
#     x_formatter=lambda x: f"experimental procedure: {x}",
#     y_name="cubic MoC weight fraction (%)",
#     y_formatter=lambda y: f"{y:.1f}",
#     model="gpt-4o",
#     selector_k=5,
#     temperature=0.7,
# )

# ── Multi-component asktell (dict x with full phase context) ──────────────────
asktell = boicl.AskTellFewShotTopk(
    prefix=(
        "The following are correctly answered questions. "
        "Each answer is numeric and ends with ###\n"
        "Note: the Mo:sucrose ratio is a variable parameter "
        "and should not be assumed constant across experiments."
    ),
    prompt_template=PromptTemplate(
        input_variables=["x", "y", "y_name"],
        template="Q: What is the {y_name} of {x}?@@@\nA: {y}###",
    ),
    suffix="What is the {y_name} of {x}?@@@\nA:",
    x_formatter=lambda x: (
    f"experimental procedure: {x['procedure']}. "
    f"Observed phases — "
    f"Mo weight fraction: {x['Mo_wf']:.1f}%, sigma: {x['Mo_sigma']:.3f}%. "
    f"Mo2C weight fraction: {x['Mo2C_wf']:.1f}%, sigma: {x['Mo2C_sigma']:.3f}%. "
    f"MoC weight fraction: {x['MoC_wf']:.1f}%, sigma: {x['MoC_sigma']:.3f}%. "
    f"MoO2 weight fraction: {x['MoO2_wf']:.1f}%, sigma: {x['MoO2_sigma']:.3f}%."
    ) if isinstance(x, dict) else x,
    y_name="cubic MoC weight fraction (%)",
    y_formatter=lambda y: f"{y:.1f}",
    x_name="synthesis procedure",
    model="gpt-4o",
    selector_k=5,
    temperature=0.7,
)

# ── Original tell loop (string x) ────────────────────────────────────────────
# for i in nonzero_indices:
#     asktell.tell(df_shuffled[PROMPT_COL].iloc[i], df_shuffled[COMPLETION_COL].iloc[i])
#     print("Told:", df_shuffled[PROMPT_COL].iloc[i], df_shuffled[COMPLETION_COL].iloc[i])

# ── Multi-component tell loop (dict x) ───────────────────────────────────────
for i in nonzero_indices:
    x = {
        'procedure':   df_shuffled['Procedure'].iloc[i],
        'Mo_wf':       float(df_shuffled['Mo (Im3m) weight fraction '].iloc[i] or 0.0),
        'Mo_sigma':    float(df_shuffled['sigma'].iloc[i] or 0.0),
        'Mo2C_wf':     float(df_shuffled['Mo2C(pbcn) weight fraction'].iloc[i] or 0.0),
        'Mo2C_sigma':  float(df_shuffled['sigma.1'].iloc[i] or 0.0),
        'MoC_wf':      float(df_shuffled['MoC(Fm3m) weight fraction'].iloc[i] or 0.0),
        'MoC_sigma':   float(df_shuffled['sigma.2'].iloc[i] or 0.0),
        'MoO2_wf':     float(df_shuffled['MoO2(P21/c) weight fraction'].iloc[i] or 0.0),
        'MoO2_sigma':  float(df_shuffled['sigma.3'].iloc[i] or 0.0),
    }
    y = float(df_shuffled['MoC(Fm3m) weight fraction'].iloc[i])
    asktell.tell(x, y)
    print("Told:", x['procedure'], y)

✅ System messages loaded
  pred: 587 chars
  inv:  3552 chars
Told: 1 g of ammonium heptamolybdate tetrahydrate dissolved in 5.5 mL of DI water. Separately, 1 g of sucrose dissolved in 2 mL of DI water at 50 °C. The two solutions were then combined. The combined solution was dried at 120 °C in air for 24 hours, then crushed and sieved to 212 microns. 0.6 g of the sieved precursor was carburized at a ramp rate of 15 °C/min to 620 °C for 0.5 hr under 30 sccm of H2 gas. After carburization the sample was cooled under 30 sccm N2 and passivated under 30 sccm 1% O2/N2 for 2 hours. 91.7
Told: 1 g of ammonium heptamolybdate tetrahydrate dissolved in 5.5 mL of DI water. Separately, 1 g of sucrose dissolved in 2 mL of DI water at 50 °C. The two solutions were then combined. The combined solution was dried at 120 °C in air for 24 hours, then crushed and sieved to 212 microns. 0.6 g of the sieved precursor was carburized at a ramp rate of 10 °C/min to 620 °C for 10 hr under 60 sccm of N2 gas. Afte

## Remove Ran Experiments

In [7]:
raw_data = df_shuffled.drop(nonzero_indices)
print("New dataset size:", len(raw_data))

New dataset size: 7773


In [8]:
import importlib, boicl.pool
importlib.reload(boicl.pool)
from boicl.pool import Pool
import cloudpickle, os, numpy as np, faiss, pickle

pool_path    = "mo2c_sucrose.pkl"
emb_path     = "mo2c_sucrose_emb.npy"
index_path   = "mo2c_sucrose_index.faiss"
prompts_path = "mo2c_sucrose_prompts.pkl"

prompts = raw_data['Procedure'].tolist()

if os.path.exists(prompts_path) and os.path.exists(emb_path) and os.path.exists(index_path):
    pool = Pool.from_prebuilt(
        prompts_path=prompts_path,
        emb_path=emb_path,
        index_path=index_path,
        formatter=lambda x: f"experimental procedure: {x}",
    )
    pool.reset()
    if len(pool._pool) != len(prompts):
        print(f"⚠️  Pool size mismatch — rebuilding ({len(prompts):,} procedures)...")
        pool = Pool(prompts, formatter=lambda x: f"experimental procedure: {x}")
        np.save(emb_path, pool._emb_matrix)
        faiss.write_index(pool.index, index_path)
        with open(prompts_path, "wb") as f:
            pickle.dump(pool._pool, f)
        print(f"✅ Pool rebuilt and saved ({len(prompts):,} procedures)")
    else:
        print(f"✅ Loaded pool from prebuilt files")
else:
    pool = Pool(prompts, formatter=lambda x: f"experimental procedure: {x}")
    np.save(emb_path, pool._emb_matrix)
    faiss.write_index(pool.index, index_path)
    with open(prompts_path, "wb") as f:
        pickle.dump(pool._pool, f)
    print(f"✅ Pool built and saved ({len(prompts):,} procedures)")

print(f"Pool size: {len(pool):,}")

⚠️  Pool size mismatch — rebuilding (7,773 procedures)...
✅ Loaded cache (10,276 embeddings).
✅ Pool rebuilt and saved (7,773 procedures)
Pool size: 7,773


In [9]:
[new_prompt1], [aq_value], [mean], [std] = asktell.ask(
    pool,
    aq_fxn="expected_improvement",
    system_message=system_message,
    inv_system_message=inv_system_message,
)

# Remove from pool so it won't be suggested again
try:
    pool.choose(new_prompt1)
except ValueError:
    pass  # not in pool (inv_predict generated an unseen procedure)

# Look up by procedure string
match = df_shuffled[df_shuffled['Procedure'] == new_prompt1]

if len(match) > 0:
    print(match.iloc[0])
else:
    print("Procedure not found in df (may be a generated/unseen suggestion):")
    print(new_prompt1)

print(
    f"\033[1mAcquisition value:\033[0m {aq_value:.3f}\n"
    f"\033[1mMean prediction:\033[0m {mean:.3f}\n"
    f"\033[1mStd dev:\033[0m {std:.3f}"
)

Procedure                      1 g of ammonium heptamolybdate tetrahydrate di...
Carburize Temp (°C)                                                          620
Ramp Rate (°C/min)                                                            10
Purge Gas                                                                     H2
AMT:Sucrose Ratio                                                            0.5
Hold Time (hr)                                                              10.0
Flow Rate (sccm)                                                             100
Mo (Im3m) weight fraction                                                    NaN
sigma                                                                        NaN
Mo2C(pbcn) weight fraction                                                   NaN
sigma.1                                                                      NaN
MoC(Fm3m) weight fraction                                                    NaN
sigma.2                     

In [10]:
[new_prompt1]

['1 g of ammonium heptamolybdate tetrahydrate dissolved in 4 mL of DI water. Separately, 2 g of sucrose dissolved in 3.5 mL of DI water at 50 °C. The two solutions were then combined. The combined solution was dried at 120 °C in air for 24 hours, then crushed and sieved to 212 microns. 0.6 g of the sieved precursor was carburized at a ramp rate of 10 °C/min to 620 °C for 10 hr under 100 sccm of H2 gas. After carburization the sample was cooled under 30 sccm N2 and passivated under 30 sccm 1% O2/N2 for 2 hours.']

## T1 current states

In [ ]:
import pandas as pd
import numpy as np
import re

# ── Parse synthesis params directly from procedure string ─────────────────────
def parse_params(proc):
    ramp  = re.search(r'ramp rate of ([\d.]+)', proc)
    temp  = re.search(r'to ([\d.]+) °C', proc)
    hold  = re.search(r'for ([\d.]+) hr', proc)
    gas   = re.search(r'under ([\d.]+ sccm of \w+) gas', proc)
    mo_g  = re.search(r'([\d.]+) g of ammonium heptamolybdate', proc)
    suc_g = re.search(r'([\d.]+) g of sucrose', proc)
    ratio_str = (
        f'{float(mo_g.group(1)):.1f} : {float(suc_g.group(1)):.1f}'
        if mo_g and suc_g else '—'
    )
    return {
        'Ramp\n(°C/min)':  float(ramp.group(1)) if ramp else None,
        'Hold Temp\n(°C)': float(temp.group(1)) if temp else None,
        'Hold Time\n(hr)': float(hold.group(1)) if hold else None,
        'Gas / Flow':      gas.group(1)          if gas  else None,
        'AMT:Suc\nRatio':  ratio_str,
    }

# ── Split seeds vs completed BO points ────────────────────────────────────────
bo_procedures = {row[0] for row in completed_prompt_labels}

seed_rows, bo_rows = [], []
for i in nonzero_indices:
    row  = df_shuffled.iloc[i]
    proc = row['Procedure']
    entry = {
        **parse_params(proc),
        'Cubic MoC\nwf (%)': float(row['MoC(Fm3m) weight fraction']),
        '_proc':              proc,
    }
    if proc in bo_procedures:
        bo_rows.append(entry)
    else:
        seed_rows.append(entry)

for k, d in enumerate(seed_rows):
    d['Label']      = ['i', 'ii', 'iii', 'iv', 'v'][k]
    d['Trajectory'] = 'Seed'

# reverse so latest run (added last to completed_prompt_labels) = label 1
bo_rows_labeled = list(reversed(bo_rows))
for k, d in enumerate(bo_rows_labeled):
    d['Label']      = str(k + 1)
    d['Trajectory'] = 'Traj 1 (EI)'

# ── Next suggested point (not yet measured) ───────────────────────────────────
next_point = {
    'Label':             'next →',
    'Trajectory':        'Traj 1 (EI)',
    **parse_params(new_prompt1),
    'Cubic MoC\nwf (%)': np.nan,
    '_proc':             new_prompt1,
}

# ── Assemble ──────────────────────────────────────────────────────────────────
all_rows    = seed_rows + bo_rows_labeled + [next_point]
full_df     = pd.DataFrame(all_rows).set_index('Label')
proc_series = full_df['_proc']

summary = full_df[[
    'Trajectory', 'Ramp\n(°C/min)', 'Hold Temp\n(°C)',
    'Hold Time\n(hr)', 'Gas / Flow', 'AMT:Suc\nRatio', 'Cubic MoC\nwf (%)'
]]

# ── Styling ───────────────────────────────────────────────────────────────────
def color_traj(val):
    return {
        'Seed':        'background-color: #d0e8ff; color: #1a3a5c',
        'Traj 1 (EI)': 'background-color: #d4edda; color: #155724',
    }.get(str(val), '')

def bar_moc(val):
    try:
        pct = float(val) / 100
        return (
            f'background: linear-gradient(90deg, #4e91d2 {pct*100:.0f}%, '
            f'transparent {pct*100:.0f}%); color: #000; font-weight: bold'
        )
    except (TypeError, ValueError):
        return 'color: #999; font-style: italic'

def highlight_next(row):
    if row.name == 'next →':
        return ['border-top: 2px dashed #e67e22; color: #e67e22; font-weight: bold'] * len(row)
    return [''] * len(row)

styled = (
    summary.style
    .map(color_traj, subset=['Trajectory'])
    .map(bar_moc,    subset=['Cubic MoC\nwf (%)'])
    .apply(highlight_next, axis=1)
    .format({
        'Cubic MoC\nwf (%)': lambda v: f'{float(v):.1f}' if pd.notna(v) else '⏳ pending',
        'AMT:Suc\nRatio':     lambda v: str(v)     if pd.notna(v) else '—',
        'Ramp\n(°C/min)':     lambda v: f'{v:.0f}' if pd.notna(v) else '—',
        'Hold Temp\n(°C)':    lambda v: f'{v:.0f}' if pd.notna(v) else '—',
        'Hold Time\n(hr)':    lambda v: f'{v:.1f}' if pd.notna(v) else '—',
        'Gas / Flow':         lambda v: str(v)      if pd.notna(v) else '—',
    })
    .set_caption('BOICL Campaign — Traj 1 (EI)  |  Cubic MoC (Fm3̄m) Optimization')
    .set_table_styles([
        {'selector': 'caption',
         'props': [('font-size', '14px'), ('font-weight', 'bold'),
                   ('text-align', 'left'), ('padding', '0 0 8px 0')]},
        {'selector': 'th',
         'props': [('background-color', '#2c5f8a'), ('color', 'white'),
                   ('padding', '8px 12px'), ('text-align', 'center'),
                   ('white-space', 'pre-line'), ('line-height', '1.3')]},
        {'selector': 'td',
         'props': [('padding', '6px 12px'), ('text-align', 'center'),
                   ('border-bottom', '1px solid #e0e0e0')]},
        {'selector': 'tr:hover td',
         'props': [('filter', 'brightness(0.95)')]},
    ])
)

display(styled)

# ── Summary stats ─────────────────────────────────────────────────────────────
measured  = summary['Cubic MoC\nwf (%)'].apply(pd.to_numeric, errors='coerce').dropna()
best_val  = measured.max()
best_lbl  = measured.idxmax()
best_traj = summary.loc[best_lbl, 'Trajectory']

print(f"\n🏆 Best so far: {best_val:.1f}%  (point {best_lbl} — {best_traj})")

# ── Full procedure strings ────────────────────────────────────────────────────
print("\n📋 Procedure strings:")
for label, proc in proc_series.items():
    moc     = summary.loc[label, 'Cubic MoC\nwf (%)']
    moc_str = f"{float(moc):.1f}%" if pd.notna(moc) else "⏳ pending"
    traj    = summary.loc[label, 'Trajectory']
    print(f"\n  [{label}]  {traj}  —  {moc_str}")
    print(f"  {proc}")

# Trajectory test (greedy)

In [6]:
import pandas as pd

DATA_PATH  = "./Metal_Sucrose_DesignSpace_fr_T2.xlsx"
SHEET_NAME = "Design Space"

LABEL_COLS = ['Mo (Im3m) weight fraction ', 'sigma', 'Mo2C(pbcn) weight fraction', 'sigma.1',
              'MoC(Fm3m) weight fraction', 'sigma.2', 'Rwp', 'GOF', 'X^2',
              'MoO2(P21/c) weight fraction', 'sigma.3', 'MoC Ratio']

df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)

# a row "has labels" if any label cell is non-empty
has_label = df[LABEL_COLS].apply(
    lambda c: c.notna() & (c.astype(str).str.strip() != "")
).any(axis=1)

df.loc[has_label, LABEL_COLS] = None
print(f"Cleared labels on {int(has_label.sum())} row(s); rows kept: {len(df)}")

with pd.ExcelWriter(DATA_PATH, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    df.to_excel(writer, sheet_name=SHEET_NAME, index=False)

Cleared labels on 5 row(s); rows kept: 7779


In [7]:
import pandas as pd
import numpy as np
import time

start_time = time.time()
RANDOM_SEED = 616
np.random.seed(RANDOM_SEED)

# Constants for column names
PROMPT_COL     = "Procedure"
COMPLETION_COL = "MoC(Fm3m) weight fraction"

# ── New results to write in each BO iteration ─────────────────────────────────
# Format: (procedure_string, Mo_wf, Mo_sigma, Mo2C_wf, Mo2C_sigma,
#           MoC_wf, MoC_sigma, MoO2_wf, MoO2_sigma,
#           Rwp, GOF, chi2, MoC_ratio)
completed_prompt_labels = [
    ("1 g of ammonium heptamolybdate tetrahydrate dissolved in 4 mL of DI water. Separately, 2 g of sucrose dissolved in 3.5 mL of DI water at 50 °C.The two solutions were then combined. The combined solution was dried at 120 °C in air for 24 hours, then crushed and sieved to 212 microns. 0.6 g of the sieved precursor was carburized at a ramp rate of 15 °C/min to 830 °C for 2 hr under 60 sccm of H2 gas. After carburization the sample was cooled under 30 sccm N2 and passivated under 30 sccm 1% O2/N2 for 2 hours.",
     0.0, 0.0,           # Mo wf, sigma
     83.4, 1.67,           # Mo2C wf, sigma
     8.3, 1.00,         # MoC wf, sigma
     8.4, 1.00,         # MoO2 wf, sigma
     10.29, 15.9, 251.6,  # Rwp, GOF, chi2
     "MoC.66"),("1 g of ammonium heptamolybdate tetrahydrate dissolved in 5.5 mL of DI water. Separately, 1 g of sucrose dissolved in 2 mL of DI water at 50 °C. The two solutions were then combined. The combined solution was dried at 120 °C in air for 24 hours, then crushed and sieved to 212 microns. 0.6 g of the sieved precursor was carburized at a ramp rate of 10 °C/min to 850 °C for 0.5 hr under 30 sccm of N2 gas. After carburization the sample was cooled under 30 sccm N2 and passivated under 30 sccm 1% O2/N2 for 2 hours.",0.0, 0.0,           # Mo wf, sigma
     0.0, 0.0,           # Mo2C wf, sigma
     91.7, 1.83,         # MoC wf, sigma
     8.3, 1.83,         # MoO2 wf, sigma
     7, 11.032, 121.71,  # Rwp, GOF, chi2
     "MoC.66"),
     ("1 g of ammonium heptamolybdate tetrahydrate dissolved in 5.5 mL of DI water. Separately, 1 g of sucrose dissolved in 2 mL of DI water at 50 °C. The two solutions were then combined. The combined solution was dried at 120 °C in air for 24 hours, then crushed and sieved to 212 microns. 0.6 g of the sieved precursor was carburized at a ramp rate of 10 °C/min to 620 °C for 0.5 hr under 60 sccm of N2 gas. After carburization the sample was cooled under 30 sccm N2 and passivated under 30 sccm 1% O2/N2 for 2 hours.",0.0, 0.0,           # Mo wf, sigma
     0.0, 0.0,           # Mo2C wf, sigma
     30.3, 1.29,         # MoC wf, sigma
     69.7, 1.39,         # MoO2 wf, sigma
     6.61, 10.4, 108.24,  # Rwp, GOF, chi2
     "MoC.66")
]

# Load dataset
data_path = "./Metal_Sucrose_DesignSpace_fr_T2.xlsx"
df = pd.read_excel(data_path, sheet_name="Design Space")

# ── Add new columns if they don't exist ───────────────────────────────────────
for col in ["MoO2(P21/c) weight fraction", "sigma.3", "MoC Ratio"]:
    if col not in df.columns:
        df[col] = None
        print(f"➕ Added column: {col}")

# ── Write new results ─────────────────────────────────────────────────────────
for row in completed_prompt_labels:
    (procedure, mo_wf, mo_sig, mo2c_wf, mo2c_sig,
     moc_wf, moc_sig, moo2_wf, moo2_sig,
     rwp, gof, chi2, moc_ratio) = row
    match = df[df[PROMPT_COL] == procedure]
    if not match.empty:
        idx = match.index[0]
        df.at[idx, 'Mo (Im3m) weight fraction ']     = mo_wf
        df.at[idx, 'sigma']                           = mo_sig
        df.at[idx, 'Mo2C(pbcn) weight fraction']      = mo2c_wf
        df.at[idx, 'sigma.1']                         = mo2c_sig
        df.at[idx, 'MoC(Fm3m) weight fraction']       = moc_wf
        df.at[idx, 'sigma.2']                         = moc_sig
        df.at[idx, 'MoO2(P21/c) weight fraction']     = moo2_wf
        df.at[idx, 'sigma.3']                         = moo2_sig
        df.at[idx, 'Rwp']                             = rwp
        df.at[idx, 'GOF']                             = gof
        df.at[idx, 'X^2']                             = chi2
        df.at[idx, 'MoC Ratio']                       = moc_ratio
        print(f"✅ Updated index {idx} — MoC wf={moc_wf}, MoO2 wf={moo2_wf}, Rwp={rwp}")
    else:
        print(f"❌ Prompt not found: {procedure[:60]}...")

# Save back to xlsx
with pd.ExcelWriter(data_path, engine="openpyxl", mode="a",
                    if_sheet_exists="replace") as writer:
    df.to_excel(writer, sheet_name="Design Space", index=False)

# ── Shuffle and report non-zero entries ───────────────────────────────────────
df_shuffled = df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

nonzero_mask    = df_shuffled[COMPLETION_COL].notna() & \
                  (df_shuffled[COMPLETION_COL].astype(str).str.strip() != "")
nonzero_indices = df_shuffled.index[nonzero_mask].to_numpy()

print("Non-zero indices:", nonzero_indices[:])
print(f"Total non-zero entries: {len(nonzero_indices)}")
print(f"\n⏱️ Elapsed time: {time.time() - start_time:.2f} seconds")

❌ Prompt not found: 1 g of ammonium heptamolybdate tetrahydrate dissolved in 4 m...
✅ Updated index 6600 — MoC wf=91.7, MoO2 wf=8.3, Rwp=7
✅ Updated index 1633 — MoC wf=30.3, MoO2 wf=69.7, Rwp=6.61


/var/folders/gn/t7lkvwjx45l9y1jzv5lphf2h0000gn/T/ipykernel_4675/3612577177.py:67: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'MoC.66' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, 'MoC Ratio']                       = moc_ratio


Non-zero indices: [5846 5850]
Total non-zero entries: 2

⏱️ Elapsed time: 3.15 seconds


In [8]:
for i in nonzero_indices:
    print(df_shuffled[COMPLETION_COL][i])

30.3
91.7


In [9]:
# Ensure MoC weight fraction is numeric
df_shuffled[COMPLETION_COL] = pd.to_numeric(df_shuffled[COMPLETION_COL], errors="coerce")

# Row with the highest MoC cubic weight fraction (best observed so far)
index_max = df_shuffled[COMPLETION_COL].idxmax()
print(f"Row index with max MoC wf: {index_max}")
print(f"Max MoC wf: {df_shuffled.at[index_max, COMPLETION_COL]:.3f}")
print(f"Procedure: {df_shuffled.at[index_max, PROMPT_COL]}")
print(f"DataFrame length: {len(df_shuffled)}")

Row index with max MoC wf: 5850
Max MoC wf: 91.700
Procedure: 1 g of ammonium heptamolybdate tetrahydrate dissolved in 5.5 mL of DI water. Separately, 1 g of sucrose dissolved in 2 mL of DI water at 50 °C. The two solutions were then combined. The combined solution was dried at 120 °C in air for 24 hours, then crushed and sieved to 212 microns. 0.6 g of the sieved precursor was carburized at a ramp rate of 10 °C/min to 850 °C for 0.5 hr under 30 sccm of N2 gas. After carburization the sample was cooled under 30 sccm N2 and passivated under 30 sccm 1% O2/N2 for 2 hours.
DataFrame length: 7779


In [12]:
system_message_path     = "./BOICL_docs/pred_system_message.txt"
inv_system_message_path = "./BOICL_docs/inv_system_message.txt"

assert os.path.exists(system_message_path), f"Missing: {system_message_path}"
assert os.path.exists(inv_system_message_path), f"Missing: {inv_system_message_path}"

with open(system_message_path, "r") as f:
    system_message = f.read()

with open(inv_system_message_path, "r") as f:
    inv_system_message = f.read()

print("✅ System messages loaded")
print(f"  pred: {len(system_message)} chars")
print(f"  inv:  {len(inv_system_message)} chars")

# ── Original simple asktell (string x, no phase context) ─────────────────────
# asktell = boicl.AskTellFewShotTopk(
#     prefix="",
#     prompt_template=PromptTemplate(
#         input_variables=["x", "y", "y_name"],
#         template="Q: What is the {y_name} of {x}?@@@\nA: {y}###",
#     ),
#     suffix="What is the {y_name} of {x}?@@@\nA:",
#     x_formatter=lambda x: f"experimental procedure: {x}",
#     y_name="cubic MoC weight fraction (%)",
#     y_formatter=lambda y: f"{y:.1f}",
#     model="gpt-4o",
#     selector_k=5,
#     temperature=0.7,
# )

# ── Multi-component asktell (dict x with full phase context) ──────────────────
asktell = boicl.AskTellFewShotTopk(
    prefix=(
        "The following are correctly answered questions. "
        "Each answer is numeric and ends with ###\n"
        "Note: the Mo:sucrose ratio is a variable parameter "
        "and should not be assumed constant across experiments."
    ),
    prompt_template=PromptTemplate(
        input_variables=["x", "y", "y_name"],
        template="Q: What is the {y_name} of {x}?@@@\nA: {y}###",
    ),
    suffix="What is the {y_name} of {x}?@@@\nA:",
    x_formatter=lambda x: (
    f"experimental procedure: {x['procedure']}. "
    f"Observed phases — "
    f"Mo weight fraction: {x['Mo_wf']:.1f}%, sigma: {x['Mo_sigma']:.3f}%. "
    f"Mo2C weight fraction: {x['Mo2C_wf']:.1f}%, sigma: {x['Mo2C_sigma']:.3f}%. "
    f"MoC weight fraction: {x['MoC_wf']:.1f}%, sigma: {x['MoC_sigma']:.3f}%. "
    f"MoO2 weight fraction: {x['MoO2_wf']:.1f}%, sigma: {x['MoO2_sigma']:.3f}%."
    ) if isinstance(x, dict) else x,
    y_name="cubic MoC weight fraction (%)",
    y_formatter=lambda y: f"{y:.1f}",
    x_name="synthesis procedure",
    model="gpt-4o",
    selector_k=5,
    temperature=0.7,
)

# ── Original tell loop (string x) ────────────────────────────────────────────
# for i in nonzero_indices:
#     asktell.tell(df_shuffled[PROMPT_COL].iloc[i], df_shuffled[COMPLETION_COL].iloc[i])
#     print("Told:", df_shuffled[PROMPT_COL].iloc[i], df_shuffled[COMPLETION_COL].iloc[i])

# ── Multi-component tell loop (dict x) ───────────────────────────────────────
for i in nonzero_indices:
    x = {
        'procedure':   df_shuffled['Procedure'].iloc[i],
        'Mo_wf':       float(df_shuffled['Mo (Im3m) weight fraction '].iloc[i] or 0.0),
        'Mo_sigma':    float(df_shuffled['sigma'].iloc[i] or 0.0),
        'Mo2C_wf':     float(df_shuffled['Mo2C(pbcn) weight fraction'].iloc[i] or 0.0),
        'Mo2C_sigma':  float(df_shuffled['sigma.1'].iloc[i] or 0.0),
        'MoC_wf':      float(df_shuffled['MoC(Fm3m) weight fraction'].iloc[i] or 0.0),
        'MoC_sigma':   float(df_shuffled['sigma.2'].iloc[i] or 0.0),
        'MoO2_wf':     float(df_shuffled['MoO2(P21/c) weight fraction'].iloc[i] or 0.0),
        'MoO2_sigma':  float(df_shuffled['sigma.3'].iloc[i] or 0.0),
    }
    y = float(df_shuffled['MoC(Fm3m) weight fraction'].iloc[i])
    asktell.tell(x, y)
    print("Told:", x['procedure'], y)

✅ System messages loaded
  pred: 587 chars
  inv:  3552 chars
Told: 1 g of ammonium heptamolybdate tetrahydrate dissolved in 5.5 mL of DI water. Separately, 1 g of sucrose dissolved in 2 mL of DI water at 50 °C. The two solutions were then combined. The combined solution was dried at 120 °C in air for 24 hours, then crushed and sieved to 212 microns. 0.6 g of the sieved precursor was carburized at a ramp rate of 10 °C/min to 620 °C for 0.5 hr under 60 sccm of N2 gas. After carburization the sample was cooled under 30 sccm N2 and passivated under 30 sccm 1% O2/N2 for 2 hours. 30.3
Told: 1 g of ammonium heptamolybdate tetrahydrate dissolved in 5.5 mL of DI water. Separately, 1 g of sucrose dissolved in 2 mL of DI water at 50 °C. The two solutions were then combined. The combined solution was dried at 120 °C in air for 24 hours, then crushed and sieved to 212 microns. 0.6 g of the sieved precursor was carburized at a ramp rate of 10 °C/min to 850 °C for 0.5 hr under 30 sccm of N2 gas. Aft

In [13]:
raw_data = df_shuffled.drop(nonzero_indices)
print("New dataset size:", len(raw_data))

New dataset size: 7777


In [14]:
import importlib, boicl.pool
importlib.reload(boicl.pool)
from boicl.pool import Pool
import cloudpickle, os, numpy as np, faiss, pickle

pool_path    = "mo2c_sucrose.pkl"
emb_path     = "mo2c_sucrose_emb.npy"
index_path   = "mo2c_sucrose_index.faiss"
prompts_path = "mo2c_sucrose_prompts.pkl"

prompts = raw_data['Procedure'].tolist()

if os.path.exists(prompts_path) and os.path.exists(emb_path) and os.path.exists(index_path):
    pool = Pool.from_prebuilt(
        prompts_path=prompts_path,
        emb_path=emb_path,
        index_path=index_path,
        formatter=lambda x: f"experimental procedure: {x}",
    )
    pool.reset()
    if len(pool._pool) != len(prompts):
        print(f"⚠️  Pool size mismatch — rebuilding ({len(prompts):,} procedures)...")
        pool = Pool(prompts, formatter=lambda x: f"experimental procedure: {x}")
        np.save(emb_path, pool._emb_matrix)
        faiss.write_index(pool.index, index_path)
        with open(prompts_path, "wb") as f:
            pickle.dump(pool._pool, f)
        print(f"✅ Pool rebuilt and saved ({len(prompts):,} procedures)")
    else:
        print(f"✅ Loaded pool from prebuilt files")
else:
    pool = Pool(prompts, formatter=lambda x: f"experimental procedure: {x}")
    np.save(emb_path, pool._emb_matrix)
    faiss.write_index(pool.index, index_path)
    with open(prompts_path, "wb") as f:
        pickle.dump(pool._pool, f)
    print(f"✅ Pool built and saved ({len(prompts):,} procedures)")

print(f"Pool size: {len(pool):,}")

⚠️  Pool size mismatch — rebuilding (7,777 procedures)...
✅ Loaded cache (10,276 embeddings).
✅ Pool rebuilt and saved (7,777 procedures)
Pool size: 7,777


In [15]:
[new_prompt], [aq_value], [mean], [std] = asktell.ask(
    pool,
    aq_fxn="greedy",
    system_message=system_message,
    inv_system_message=inv_system_message,
)

# Remove from pool so it won't be suggested again
try:
    pool.choose(new_prompt)
except ValueError:
    pass  # not in pool (inv_predict generated an unseen procedure)

# Look up by procedure string
match = df_shuffled[df_shuffled['Procedure'] == new_prompt]

if len(match) > 0:
    print(match.iloc[0])
else:
    print("Procedure not found in df (may be a generated/unseen suggestion):")
    print(new_prompt)

print(
    f"\033[1mAcquisition value:\033[0m {aq_value:.3f}\n"
    f"\033[1mMean prediction:\033[0m {mean:.3f}\n"
    f"\033[1mStd dev:\033[0m {std:.3f}"
)

Procedure                      1 g of ammonium heptamolybdate tetrahydrate di...
Carburize Temp (°C)                                                          830
Ramp Rate (°C/min)                                                            15
Purge Gas                                                                     N2
AMT:Sucrose Ratio                                                            1.0
Hold Time (hr)                                                               2.0
Flow Rate (sccm)                                                              60
Mo (Im3m) weight fraction                                                    NaN
sigma                                                                        NaN
Mo2C(pbcn) weight fraction                                                   NaN
sigma.1                                                                      NaN
MoC(Fm3m) weight fraction                                                    NaN
sigma.2                     

In [16]:
new_prompt

'1 g of ammonium heptamolybdate tetrahydrate dissolved in 5.5 mL of DI water. Separately, 1 g of sucrose dissolved in 2 mL of DI water at 50 °C. The two solutions were then combined. The combined solution was dried at 120 °C in air for 24 hours, then crushed and sieved to 212 microns. 0.6 g of the sieved precursor was carburized at a ramp rate of 15 °C/min to 830 °C for 2 hr under 60 sccm of N2 gas. After carburization the sample was cooled under 30 sccm N2 and passivated under 30 sccm 1% O2/N2 for 2 hours.'

# T2 current sate

In [29]:
import pandas as pd
import numpy as np
import re

# ── T2 procedure strings ───────────────────────────────────────────────────────
t2_seed_i = (
    "1 g of ammonium heptamolybdate tetrahydrate dissolved in 4 mL of DI water. "
    "Separately, 2 g of sucrose dissolved in 3.5 mL of DI water at 50 °C."
    "The two solutions were then combined. The combined solution was dried at 120 °C "
    "in air for 24 hours, then crushed and sieved to 212 microns. 0.6 g of the sieved "
    "precursor was carburized at a ramp rate of 15 °C/min to 830 °C for 2 hr under "
    "60 sccm of H2 gas. After carburization the sample was cooled under 30 sccm N2 "
    "and passivated under 30 sccm 1% O2/N2 for 2 hours."
)

t2_seed_ii = (
    "1 g of ammonium heptamolybdate tetrahydrate dissolved in 5.5 mL of DI water. "
    "Separately, 1 g of sucrose dissolved in 2 mL of DI water at 50 °C. "
    "The two solutions were then combined. The combined solution was dried at 120 °C "
    "in air for 24 hours, then crushed and sieved to 212 microns. 0.6 g of the sieved "
    "precursor was carburized at a ramp rate of 10 °C/min to 850 °C for 0.5 hr under "
    "30 sccm of N2 gas. After carburization the sample was cooled under 30 sccm N2 "
    "and passivated under 30 sccm 1% O2/N2 for 2 hours."
)

# ── Parse synthesis params from procedure string ──────────────────────────────
def parse_params(proc):
    ramp  = re.search(r'ramp rate of ([\d.]+)', proc)
    temp  = re.search(r'to ([\d.]+) °C', proc)
    hold  = re.search(r'for ([\d.]+) hr', proc)
    gas   = re.search(r'under ([\d.]+ sccm of \w+) gas', proc)
    mo_g  = re.search(r'([\d.]+) g of ammonium heptamolybdate', proc)
    suc_g = re.search(r'([\d.]+) g of sucrose', proc)
    ratio_str = (
        f'{float(mo_g.group(1)):.1f} : {float(suc_g.group(1)):.1f}'
        if mo_g and suc_g else '—'
    )
    return {
        'Ramp\n(°C/min)':  float(ramp.group(1)) if ramp else None,
        'Hold Temp\n(°C)': float(temp.group(1)) if temp else None,
        'Hold Time\n(hr)': float(hold.group(1)) if hold else None,
        'Gas / Flow':      gas.group(1)          if gas  else None,
        'AMT:Suc\nRatio':  ratio_str,
    }

# ── Build rows ────────────────────────────────────────────────────────────────
seed_rows = []
for label, proc in [('i', t2_seed_i), ('ii', t2_seed_ii)]:
    seed_rows.append({
        'Label':             label,
        'Trajectory':        'Seed',
        **parse_params(proc),
        'Cubic MoC\nwf (%)': np.nan,
        '_proc':             proc,
    })

# T2 BO point — greedy suggestion, not yet measured
# Update label, proc, and wf as new results come in; add more dicts to bo_rows
bo_rows = [
    {
        'Label':             '1',
        'Trajectory':        'Traj 2 (Greedy)',
        **parse_params(new_prompt),   # new_prompt is the greedy suggestion variable
        'Cubic MoC\nwf (%)': np.nan,
        '_proc':             new_prompt,
    },
]

# ── Next suggested point placeholder ─────────────────────────────────────────
# Remove or replace this block once T2 generates a second suggestion
next_point = None   # set to a dict like bo_rows entries when available

# ── Assemble ──────────────────────────────────────────────────────────────────
all_rows    = seed_rows + bo_rows + ([next_point] if next_point else [])
full_df     = pd.DataFrame(all_rows).set_index('Label')
proc_series = full_df['_proc']

summary = full_df[[
    'Trajectory', 'Ramp\n(°C/min)', 'Hold Temp\n(°C)',
    'Hold Time\n(hr)', 'Gas / Flow', 'AMT:Suc\nRatio', 'Cubic MoC\nwf (%)'
]]

# ── Styling ───────────────────────────────────────────────────────────────────
def color_traj(val):
    return {
        'Seed':             'background-color: #d0e8ff; color: #1a3a5c',
        'Traj 2 (Greedy)':  'background-color: #fde8d0; color: #7a3000',
    }.get(str(val), '')

def bar_moc(val):
    try:
        pct = float(val) / 100
        return (
            f'background: linear-gradient(90deg, #e07b3a {pct*100:.0f}%, '
            f'transparent {pct*100:.0f}%); color: #000; font-weight: bold'
        )
    except (TypeError, ValueError):
        return 'color: #999; font-style: italic'

def highlight_pending(row):
    if pd.isna(row.get('Cubic MoC\nwf (%)')) and row.name != 'next →':
        return ['border-left: 3px solid #e67e22'] + [''] * (len(row) - 1)
    if row.name == 'next →':
        return ['border-top: 2px dashed #e67e22; color: #e67e22; font-weight: bold'] * len(row)
    return [''] * len(row)

styled = (
    summary.style
    .map(color_traj, subset=['Trajectory'])
    .map(bar_moc,    subset=['Cubic MoC\nwf (%)'])
    .apply(highlight_pending, axis=1)
    .format({
        'Cubic MoC\nwf (%)': lambda v: f'{float(v):.1f}' if pd.notna(v) else '⏳ pending',
        'AMT:Suc\nRatio':     lambda v: str(v)     if pd.notna(v) else '—',
        'Ramp\n(°C/min)':     lambda v: f'{v:.0f}' if pd.notna(v) else '—',
        'Hold Temp\n(°C)':    lambda v: f'{v:.0f}' if pd.notna(v) else '—',
        'Hold Time\n(hr)':    lambda v: f'{v:.1f}' if pd.notna(v) else '—',
        'Gas / Flow':         lambda v: str(v)      if pd.notna(v) else '—',
    })
    .set_caption('BOICL Campaign — Traj 2 (Greedy)  |  Cubic MoC (Fm3̄m) Optimization')
    .set_table_styles([
        {'selector': 'caption',
         'props': [('font-size', '14px'), ('font-weight', 'bold'),
                   ('text-align', 'left'), ('padding', '0 0 8px 0')]},
        {'selector': 'th',
         'props': [('background-color', '#7a3000'), ('color', 'white'),
                   ('padding', '8px 12px'), ('text-align', 'center'),
                   ('white-space', 'pre-line'), ('line-height', '1.3')]},
        {'selector': 'td',
         'props': [('padding', '6px 12px'), ('text-align', 'center'),
                   ('border-bottom', '1px solid #e0e0e0')]},
        {'selector': 'tr:hover td',
         'props': [('filter', 'brightness(0.95)')]},
    ])
)

display(styled)

# ── Summary stats (only over measured points) ─────────────────────────────────
measured = summary['Cubic MoC\nwf (%)'].apply(pd.to_numeric, errors='coerce').dropna()

if len(measured) > 0:
    best_val  = measured.max()
    best_lbl  = measured.idxmax()
    best_traj = summary.loc[best_lbl, 'Trajectory']
    print(f"\n🏆 Best so far: {best_val:.1f}%  (point {best_lbl} — {best_traj})")
else:
    print("\n⏳ No measurements yet — all points pending.")

# ── Full procedure strings ────────────────────────────────────────────────────
print("\n📋 Procedure strings:")
for label, proc in proc_series.items():
    moc     = summary.loc[label, 'Cubic MoC\nwf (%)']
    moc_str = f"{float(moc):.1f}%" if pd.notna(moc) else "⏳ pending"
    traj    = summary.loc[label, 'Trajectory']
    print(f"\n  [{label}]  {traj}  —  {moc_str}")
    print(f"  {proc}")

,Trajectory,Ramp (°C/min),Hold Temp (°C),Hold Time (hr),Gas / Flow,AMT:Suc Ratio,Cubic MoC wf (%)
Label,,,,,,,
i,Seed,15,830,2.0,60 sccm of H2,1.0 : 2.0,⏳ pending
ii,Seed,10,850,0.5,30 sccm of N2,1.0 : 1.0,⏳ pending
1,Traj 2 (Greedy),10,630,2.0,60 sccm of H2,1.0 : 1.0,⏳ pending



⏳ No measurements yet — all points pending.

📋 Procedure strings:

  [i]  Seed  —  ⏳ pending
  1 g of ammonium heptamolybdate tetrahydrate dissolved in 4 mL of DI water. Separately, 2 g of sucrose dissolved in 3.5 mL of DI water at 50 °C.The two solutions were then combined. The combined solution was dried at 120 °C in air for 24 hours, then crushed and sieved to 212 microns. 0.6 g of the sieved precursor was carburized at a ramp rate of 15 °C/min to 830 °C for 2 hr under 60 sccm of H2 gas. After carburization the sample was cooled under 30 sccm N2 and passivated under 30 sccm 1% O2/N2 for 2 hours.

  [ii]  Seed  —  ⏳ pending
  1 g of ammonium heptamolybdate tetrahydrate dissolved in 5.5 mL of DI water. Separately, 1 g of sucrose dissolved in 2 mL of DI water at 50 °C. The two solutions were then combined. The combined solution was dried at 120 °C in air for 24 hours, then crushed and sieved to 212 microns. 0.6 g of the sieved precursor was carburized at a ramp rate of 10 °C/min to 85